# WAIND 0.8 — consenso, viés de participante e retreino

Notebook complementar ao `waind_experimento_intensidade_colab.ipynb`. Ele **não substitui** os experimentos
de lá; acrescenta três peças que aproximam a estimativa da intensidade "real" de um odor:

1. **Rótulo por consenso.** Relatos próximos no espaço e no tempo (padrão: 100 m e 120 min, o mesmo do
   exportador) formam um *grupo de ocorrência*. O rótulo do grupo combina as notas de todos que estavam lá,
   com uma medida de dispersão (incerteza).
2. **Correção do viés de cada participante.** Algumas pessoas tendem a dar notas mais altas ou mais baixas.
   O notebook estima esse desvio, com encolhimento para zero quando a pessoa tem poucos relatos, e calcula a
   *intensidade ajustada* (nota menos o viés estimado).
3. **Regras de retreino.** Só treina com dados mínimos, só troca o modelo vigente quando o novo supera a
   referência (mediana) com margem, e guarda cada versão com métricas e hash do dataset.

**Modo sintético (padrão):** gera dados com padrões conhecidos — esgoto mais forte que lixo, vento forte
dilui o odor, calor intensifica, um participante que exagera — e verifica se o pipeline **recupera** esses
padrões. Isso prova que o pipeline aprende; não é evidência sobre Santa Maria.

**Modo campo:** lê a exportação privada (`registros.jsonl` + `manifesto.json`) gerada por
`exportar-dataset-pesquisa.mjs`. O notebook e os dados devem permanecer privados.

A nota humana nunca é alterada; a intensidade ajustada e as previsões ficam em arquivos separados.

In [ ]:
# Configuração. Em campo: MODO = "campo" e DATASET_DIR apontando para a pasta exportada no Drive.
from pathlib import Path
import sys, json, math, hashlib, datetime as dt
import numpy as np
import pandas as pd

MODO = "sintetico"            # "sintetico" ou "campo"
EXPORTAR = True               # modo campo: lê o Firebase agora (True) ou usa uma exportação já feita (False)
PROJETO_FIREBASE = "waind-ufn-dev"
SEED = 42
EM_COLAB = "google.colab" in sys.modules
BASE = Path("/content/drive/MyDrive/WAIND") if EM_COLAB else Path("waind_local")
DATASET_DIR = BASE / "datasets" / "AAAA-MM-DD"   # usado só com EXPORTAR = False
SAL = BASE / "privado" / "sal_pseudonimos.bin"    # nunca apagar nem compartilhar (ver célula de exportação)
MODELOS_DIR = BASE / "modelos"

# Agrupamento (igual ao exportador) e regras de retreino.
GRUPO_METROS, GRUPO_MINUTOS = 100, 120
MIN_RELATOS = 100              # relatos elegíveis
MIN_PARTICIPANTES = 15         # pseudônimos distintos
MIN_GRUPOS = 20                # grupos de ocorrência
MIN_GRUPOS_MULTI = 8           # grupos com 2+ participantes (base do viés e do consenso)
ENCOLHIMENTO_VIES = 2.0        # quantos "relatos fictícios de viés zero" cada pessoa recebe
MARGEM_MELHORA = 0.05          # MAE em níveis: o candidato precisa ser pelo menos isto melhor que a mediana
FRACAO_DOBRAS_MELHORA = 0.6    # e vencer a mediana nesta fração das dobras
TOLERANCIA_VIGENTE = 0.05      # não troca se ficar pior que o MAE registrado do vigente além disto

if EM_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
rng = np.random.default_rng(SEED)
print("Modo:", MODO, "| modelos em", MODELOS_DIR)

## Gerador sintético com padrões conhecidos

Cada **fonte** (contentor, bueiro, vala…) tem uma força de base pela categoria. Cada **episódio** soma o
efeito do clima. Cada **participante** tem um viés pessoal; um deles exagera de propósito (+1,2). A nota
observada é a intensidade latente + viés + ruído, arredondada para 1–5. A "verdade" fica separada das
linhas, que seguem exatamente o formato do exportador.

In [ ]:
CATEGORIAS = ["waste", "container", "sewer", "ditch", "organic", "metabolic", "chemical", "other"]
BASE_CATEGORIA = {"waste": 2.0, "container": 2.2, "organic": 2.6, "ditch": 3.0, "sewer": 3.4}
EFEITO_VENTO = -0.035          # por km/h: vento forte dilui
EFEITO_CALOR = 0.05            # por °C acima de 20

def gerar_sintetico(n_fontes=45, episodios_por_fonte=4, n_participantes=30, seed=SEED):
    r = np.random.default_rng(seed)
    vies = r.normal(0, 0.45, n_participantes); vies[0] = 1.2   # participante 0 exagera
    vies = vies - vies.mean()
    pseud = [hashlib.sha256(f"sintetico-{j}".encode()).hexdigest() for j in range(n_participantes)]
    rows, verdade_grupo = [], {}
    inicio = dt.datetime(2026, 12, 4, 7, 0)
    for f in range(n_fontes):
        cat = list(BASE_CATEGORIA)[f % len(BASE_CATEGORIA)]
        lat0, lon0 = -29.690 + r.uniform(-0.012, 0.012), -53.812 + r.uniform(-0.012, 0.012)
        for e in range(episodios_por_fonte):
            quando = inicio + dt.timedelta(days=int(r.integers(0, 28)), hours=int(r.integers(0, 13)), minutes=int(r.integers(0, 60)))
            vento, temp = float(r.uniform(0, 30)), float(r.uniform(14, 34))
            direcao = float(r.uniform(0, 360))
            latente = BASE_CATEGORIA[cat] + EFEITO_VENTO * vento + EFEITO_CALOR * (temp - 20) + r.normal(0, 0.25)
            chave = f"f{f}-e{e}"; verdade_grupo[chave] = latente
            for j in r.choice(n_participantes, size=1 + int(r.poisson(1.6)), replace=False):
                nota = int(np.clip(np.rint(latente + vies[j] + r.normal(0, 0.45)), 1, 5))
                t = quando + dt.timedelta(minutes=int(r.integers(0, 40)))
                d = math.radians(direcao)
                rows.append({
                    "alvo": {"intensidade_humana": nota},
                    "entradas_estruturadas": {
                        "categorias": [cat], "multi_hot": {c: int(c == cat) for c in CATEGORIAS},
                        "visivel_por_perto": [], "sem_foco_visivel": False,
                        "data": t.strftime("%Y-%m-%d"), "hora": t.strftime("%H:%M:00"), "tipo_ocorrencia": "atual",
                        "latitude": lat0 + r.normal(0, 0.00012), "longitude": lon0 + r.normal(0, 0.00012),
                        "clima_valido_para_ocorrencia": True,
                        "clima": {"temperatura_c": temp, "umidade_pct": float(r.uniform(40, 90)), "vento_kmh": vento,
                                  "rajada_kmh": vento * 1.8, "vento_dir_sin": math.sin(d), "vento_dir_cos": math.cos(d),
                                  "condicao_codigo": 1}},
                    "entradas_texto": {"outro_descricao": "", "observacao": "", "texto_original": "", "texto_sem_indicio_direto": ""},
                    "entrada_visual": {"caminho": None, "largura": None, "altura": None, "tem_foto": False},
                    "controle_coleta_e_divisao": {
                        "id": hashlib.sha256(f"{chave}-{j}".encode()).hexdigest()[:32], "participante_pseudo_id": pseud[j],
                        "grupo_ocorrencia_ou_foto": None, "origem_dataset": "sintetico-pipeline", "tem_foto": False,
                        "_verdade_episodio": chave}})
    verdade = {"vies": dict(zip(pseud, vies)), "exagerador": pseud[0], "grupo_latente": verdade_grupo}
    return rows, verdade

## Exportar do Firebase (modo campo)

1. Você entra com a **sua conta Google** (a dona do projeto Firebase). Nenhuma chave de serviço é criada
   ou guardada.
2. O notebook lê `relatos_pesquisa`, `relatos_mapa` e `relatos_fotos` e grava uma cópia temporária só na
   máquina do Colab (ela tem nomes e e-mails, então é apagada no fim da célula).
3. Roda o exportador oficial do repositório, **fixado num commit e conferido por SHA-256**, que valida tudo,
   exclui os testes, pseudonimiza os participantes e grava o dataset em `WAIND/datasets/<data>/`.

**O sal dos pseudônimos.** Na primeira exportação é criado `WAIND/privado/sal_pseudonimos.bin`. Ele faz o
mesmo participante ter sempre o mesmo pseudônimo; se for apagado ou trocado, a correção de viés passa a
tratar as mesmas pessoas como novas. Faça uma cópia de segurança dele e não compartilhe.

In [ ]:
import base64, os, shutil, subprocess, urllib.request
COMMIT_EXPORTADOR = "304766b62629077a4af963d65e652b72b2cab41f"
SHA_EXPORTADOR = "15b3913568d215e8713485d80c0c029730def212846845ed5077dfdf343a1482"
REPO_RAW = "https://raw.githubusercontent.com/Murillosvrj/mau-odor-ufn"

def valor_simples(v):
    """Converte tipos do Firestore (data/hora, bytes, referências) em JSON simples."""
    if isinstance(v, dict): return {k: valor_simples(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)): return [valor_simples(x) for x in v]
    if isinstance(v, dt.datetime): return v.isoformat()
    if isinstance(v, bytes): return base64.b64encode(v).decode()
    if hasattr(v, "path") and hasattr(v, "id"): return v.path
    return v

def ler_colecoes(db):
    return {c: {d.id: valor_simples(d.to_dict()) for d in db.collection(c).stream()}
            for c in ["relatos_pesquisa", "relatos_mapa", "relatos_fotos"]}

def preparar_exportador(pasta, baixar=urllib.request.urlopen):
    (pasta / "pesquisa").mkdir(parents=True, exist_ok=True)
    codigo = baixar(f"{REPO_RAW}/{COMMIT_EXPORTADOR}/pesquisa/exportar-dataset-pesquisa.mjs").read()
    assert hashlib.sha256(codigo).hexdigest() == SHA_EXPORTADOR, "Exportador diferente do conferido"
    (pasta / "pesquisa" / "exportar-dataset-pesquisa.mjs").write_bytes(codigo)
    qa = baixar(f"{REPO_RAW}/main/qa-ids.json").read()   # lista de testes sempre atualizada
    assert all(isinstance(v, list) for k, v in json.loads(qa).items() if k not in ("schema_version", "notice", "sentinel"))
    (pasta / "qa-ids.json").write_bytes(qa)
    return pasta / "pesquisa" / "exportar-dataset-pesquisa.mjs"

def exportar(colecoes, script, destino, sal, temp):
    temp.mkdir(parents=True, exist_ok=True); snap = temp / "snapshot.json"
    try:
        snap.write_text(json.dumps({"collections": colecoes}, ensure_ascii=False), encoding="utf8")
        args = ["node", str(script), "--snapshot", str(snap), "--out", str(destino), "--salt-file", str(sal)]
        if not sal.exists():
            print("Primeira exportação: criando o sal dos pseudônimos em", sal); args.append("--create-salt")
        r = subprocess.run(args, capture_output=True, text=True)
        if r.returncode: raise RuntimeError("Exportador recusou os dados: " + (r.stderr or r.stdout).strip())
        return json.loads(r.stdout.strip().splitlines()[-1])
    finally:
        shutil.rmtree(temp, ignore_errors=True)   # a cópia temporária tem nomes e e-mails

if MODO == "campo" and EXPORTAR:
    from google.colab import auth
    auth.authenticate_user()
    os.environ["GOOGLE_CLOUD_PROJECT"] = PROJETO_FIREBASE
    from google.cloud import firestore
    colecoes = ler_colecoes(firestore.Client(project=PROJETO_FIREBASE))
    print({c: len(v) for c, v in colecoes.items()}, "documentos lidos")
    script = preparar_exportador(Path("/content/waind_exportador"))
    DATASET_DIR = BASE / "datasets" / dt.datetime.now().strftime("%Y-%m-%d_%H%M")
    print(exportar(colecoes, script, DATASET_DIR, SAL, Path("/content/waind_tmp")))
    del colecoes

## Carregar os relatos

Em campo, confere o hash do manifesto, recusa mistura com dados sintéticos e exige o pseudônimo do
participante (sem ele não há como estimar viés). O exportador já excluiu os IDs de teste.

In [ ]:
def carregar_campo(pasta):
    manifesto = json.loads((pasta / "manifesto.json").read_text(encoding="utf8"))
    bruto = (pasta / "registros.jsonl").read_bytes()
    assert hashlib.sha256(bruto).hexdigest() == manifesto["sha256_registros"], "Manifesto divergente"
    linhas = [json.loads(l) for l in bruto.decode("utf8").splitlines() if l.strip()]
    assert all(l["controle_coleta_e_divisao"]["origem_dataset"] == "campo" for l in linhas), "Dados sintéticos na pasta de campo"
    return linhas, manifesto["sha256_registros"]

if MODO == "sintetico":
    linhas, VERDADE = gerar_sintetico()
    DATASET_SHA = hashlib.sha256(json.dumps(linhas, sort_keys=True).encode()).hexdigest()
else:
    linhas, DATASET_SHA = carregar_campo(DATASET_DIR); VERDADE = None
sem_pseudo = sum(1 for l in linhas if not l["controle_coleta_e_divisao"].get("participante_pseudo_id"))
linhas = [l for l in linhas if l["controle_coleta_e_divisao"].get("participante_pseudo_id")]
print(f"{len(linhas)} relatos com participante; {sem_pseudo} sem pseudônimo ficaram fora do viés/consenso")

## Grupos de ocorrência

Mesma regra do exportador: relatos a até 100 m e 120 min entram no mesmo grupo (encadeado). Se o exportador
já trouxe `grupo_ocorrencia_ou_foto`, ele é respeitado.

In [ ]:
def distancia_m(a, b):
    r = math.pi / 180; x = (a[0] - b[0]) * r; y = (a[1] - b[1]) * r
    return 6371000 * 2 * math.asin(min(1, math.sqrt(math.sin(x / 2) ** 2 + math.cos(a[0] * r) * math.cos(b[0] * r) * math.sin(y / 2) ** 2)))

def agrupar(linhas, metros=GRUPO_METROS, minutos=GRUPO_MINUTOS):
    pai = list(range(len(linhas)))
    def raiz(i):
        while pai[i] != i: pai[i] = pai[pai[i]]; i = pai[i]
        return i
    pts = [(l["entradas_estruturadas"]["latitude"], l["entradas_estruturadas"]["longitude"],
            pd.Timestamp(f'{l["entradas_estruturadas"]["data"]}T{l["entradas_estruturadas"]["hora"]}')) for l in linhas]
    ordem = sorted(range(len(linhas)), key=lambda i: pts[i][2])
    for a, i in enumerate(ordem):
        for k in ordem[a + 1:]:
            if (pts[k][2] - pts[i][2]).total_seconds() > minutos * 60: break
            if distancia_m(pts[i], pts[k]) <= metros: pai[raiz(k)] = raiz(i)
    return [f"g{raiz(i)}" for i in range(len(linhas))]

grupos = [l["controle_coleta_e_divisao"].get("grupo_ocorrencia_ou_foto") for l in linhas]
if not all(grupos): grupos = agrupar(linhas)
df = pd.DataFrame({
    "id": [l["controle_coleta_e_divisao"]["id"] for l in linhas],
    "participante": [l["controle_coleta_e_divisao"]["participante_pseudo_id"] for l in linhas],
    "grupo": grupos,
    "nota": [l["alvo"]["intensidade_humana"] for l in linhas],
})
por_grupo = df.groupby("grupo")["participante"].nunique()
print(f"{df.grupo.nunique()} grupos; {int((por_grupo >= 2).sum())} com 2+ participantes; "
      f"{df.participante.nunique()} participantes")

## Viés de participante e intensidade ajustada

Modelo aditivo: `nota = nível do grupo + viés da pessoa + ruído`. Estimado por mínimos quadrados alternados,
com encolhimento: quem tem poucos relatos fica perto de viés zero. Só grupos com duas ou mais pessoas
informam o viés (num grupo de uma pessoa só, o nível do grupo absorve tudo). A escala 1–5 é ordinal;
tratá-la como número é uma aproximação, assumida aqui por ser transparente e verificável.

In [ ]:
def estimar_vies(df, encolhimento=ENCOLHIMENTO_VIES, iteracoes=200):
    g = df.grupo.to_numpy(); p = df.participante.to_numpy(); y = df.nota.to_numpy(float)
    vies = pd.Series(0.0, index=pd.unique(p))
    for _ in range(iteracoes):
        nivel = pd.Series(y - vies.loc[p].to_numpy()).groupby(g).mean()
        resid = pd.Series(y - nivel[g].to_numpy()).groupby(p)
        novo = resid.sum() / (resid.count() + encolhimento)
        novo = novo - novo.mean()
        if np.abs(novo - vies[novo.index]).max() < 1e-9: vies = novo; break
        vies = novo
    nivel = pd.Series(y - vies.loc[p].to_numpy()).groupby(g).mean()
    return vies.rename("vies"), nivel.rename("nivel")

vies, nivel = estimar_vies(df)
df["vies_estimado"] = df.participante.map(vies)
df["intensidade_ajustada"] = (df.nota - df.vies_estimado).clip(1, 5)
consenso = df.groupby("grupo").agg(
    n_relatos=("nota", "size"), n_participantes=("participante", "nunique"),
    mediana_bruta=("nota", "median"), media_ajustada=("intensidade_ajustada", "mean"),
    dispersao_ajustada=("intensidade_ajustada", "std")).fillna({"dispersao_ajustada": 0.0})
print(consenso.describe().round(2))

### Verificação (só no modo sintético): o viés foi recuperado?

In [ ]:
if VERDADE:
    comp = pd.DataFrame({"estimado": vies, "verdadeiro": pd.Series(VERDADE["vies"])}).dropna()
    corr = comp.estimado.corr(comp.verdadeiro)
    topo = vies.idxmax() == VERDADE["exagerador"]
    print(f"Correlação viés estimado × verdadeiro: {corr:.2f} | exagerador identificado como o maior viés: {topo}")
    lat = df.assign(ep=[l["controle_coleta_e_divisao"]["_verdade_episodio"] for l in linhas]).groupby("grupo")["ep"].first().map(VERDADE["grupo_latente"])
    erro_bruto = (consenso.mediana_bruta - lat.clip(1, 5)).abs().mean()
    erro_ajust = (consenso.media_ajustada - lat.clip(1, 5)).abs().mean()
    print(f"Erro do rótulo vs latente — mediana bruta: {erro_bruto:.3f} | média ajustada: {erro_ajust:.3f}")
    VERIF = {"correlacao_vies": round(float(corr), 3), "exagerador_detectado": bool(topo),
             "erro_rotulo_bruto": round(float(erro_bruto), 3), "erro_rotulo_ajustado": round(float(erro_ajust), 3)}
else:
    VERIF = None

## Entradas do modelo (por grupo)

Cada grupo vira uma linha: categorias (união), contexto, hora (seno/cosseno), dia da semana e clima.
Nenhuma nota, nome, pseudônimo ou ID entra como entrada.

In [ ]:
def entradas_grupo(linhas, grupos):
    itens = {}
    for l, g in zip(linhas, grupos):
        e = l["entradas_estruturadas"]
        if g not in itens:
            h = int(e["hora"][:2]) + int(e["hora"][3:5]) / 60
            c = e["clima"] if e.get("clima_valido_para_ocorrencia") else {}
            itens[g] = {**{f"cat_{k}": 0.0 for k in CATEGORIAS},
                        "latitude": e["latitude"], "longitude": e["longitude"],
                        "hora_sin": math.sin(2 * math.pi * h / 24), "hora_cos": math.cos(2 * math.pi * h / 24),
                        "dia_semana": float(pd.Timestamp(e["data"]).dayofweek),
                        **{f"clima_{k}": (np.nan if c.get(k) is None else float(c[k]))
                           for k in ["temperatura_c", "umidade_pct", "vento_kmh", "rajada_kmh", "vento_dir_sin", "vento_dir_cos"]}}
        for k in e["categorias"]: itens[g][f"cat_{k}"] = 1.0
    return pd.DataFrame.from_dict(itens, orient="index")

X = entradas_grupo(linhas, df.grupo.tolist()).loc[consenso.index]
y = consenso.media_ajustada.to_numpy()
peso = np.sqrt(consenso.n_participantes.to_numpy())   # grupos com mais gente pesam mais
print("Entradas:", X.shape, list(X.columns))

## Avaliação agrupada e regra de troca

Duas divisões: por **grupo** (ocorrências novas) e por **local** (células de ~300 m inteiras ficam fora do
treino, para medir se o modelo generaliza para lugares que ainda não viu). A referência é a mediana do treino.
O candidato só é promovido se cumprir todos os mínimos e vencer a referência com margem nas duas divisões.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold

def novo_modelo():
    return make_pipeline(SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True),
                         RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=SEED, n_jobs=-1))

def avaliar(X, y, peso, rotulos_divisao, dobras=5):
    k = min(dobras, len(set(rotulos_divisao)))
    if k < 3: return None
    res = []
    for tr, te in GroupKFold(k).split(X, y, rotulos_divisao):
        m = novo_modelo().fit(X.iloc[tr], y[tr], randomforestregressor__sample_weight=peso[tr])
        mae_m = float(np.average(np.abs(np.clip(m.predict(X.iloc[te]), 1, 5) - y[te]), weights=peso[te]))
        mae_b = float(np.average(np.abs(np.median(y[tr]) - y[te]), weights=peso[te]))
        res.append({"mae_modelo": mae_m, "mae_mediana": mae_b})
    return pd.DataFrame(res)

def decidir(df, consenso, X, y, peso, vigente=None):
    req = {"relatos": (len(df), MIN_RELATOS), "participantes": (df.participante.nunique(), MIN_PARTICIPANTES),
           "grupos": (len(consenso), MIN_GRUPOS), "grupos_multi": (int((consenso.n_participantes >= 2).sum()), MIN_GRUPOS_MULTI)}
    faltando = {k: f"{v}/{m}" for k, (v, m) in req.items() if v < m}
    if faltando: return {"decisao": "nao_treinar", "motivo": f"dados insuficientes: {faltando}"}, None
    local = (np.floor(X.latitude / 0.003).astype(int).astype(str) + "_" + np.floor(X.longitude / 0.003).astype(int).astype(str)).to_numpy()
    avals = {"grupo": avaliar(X, y, peso, X.index.to_numpy()), "local": avaliar(X, y, peso, local)}
    resumo = {}
    for nome, a in avals.items():
        if a is None: return {"decisao": "nao_treinar", "motivo": f"poucos blocos para a divisão por {nome}"}, None
        resumo[nome] = {"mae_modelo": round(a.mae_modelo.mean(), 4), "mae_mediana": round(a.mae_mediana.mean(), 4),
                        "fracao_dobras_vencidas": round(float((a.mae_modelo < a.mae_mediana).mean()), 3)}
    ok = all(r["mae_mediana"] - r["mae_modelo"] >= MARGEM_MELHORA and r["fracao_dobras_vencidas"] >= FRACAO_DOBRAS_MELHORA for r in resumo.values())
    if not ok: return {"decisao": "manter_vigente", "motivo": "não superou a mediana com margem", "avaliacao": resumo}, None
    if vigente and resumo["local"]["mae_modelo"] > vigente["avaliacao"]["local"]["mae_modelo"] + TOLERANCIA_VIGENTE:
        return {"decisao": "manter_vigente", "motivo": "pior que o vigente registrado", "avaliacao": resumo}, None
    final = novo_modelo().fit(X, y, randomforestregressor__sample_weight=peso)
    return {"decisao": "promover", "motivo": "superou a mediana nas duas divisões", "avaliacao": resumo}, final

vigente_path = MODELOS_DIR / "vigente.json"
vigente = json.loads(vigente_path.read_text()) if vigente_path.exists() else None
DECISAO, MODELO = decidir(df, consenso, X, y, peso, vigente)
print(json.dumps(DECISAO, ensure_ascii=False, indent=1))

### Verificação (só no modo sintético): o modelo aprendeu os padrões?

Compara previsões trocando uma coisa de cada vez, com o resto no valor típico. Esperado: esgoto > lixo,
vento forte < vento fraco, calor > frio.

In [ ]:
def contrafactual(modelo, X, **mudar):
    base = X.median(numeric_only=True).to_frame().T
    for k in CATEGORIAS: base[f"cat_{k}"] = 0.0
    base["cat_waste"] = 1.0
    linha = base.copy()
    for k, v in mudar.items():
        if k.startswith("cat_"):
            for c in CATEGORIAS: linha[f"cat_{c}"] = 0.0
        linha[k] = v
    return float(modelo.predict(linha[X.columns])[0] - modelo.predict(base[X.columns])[0])

if VERDADE and MODELO is not None:
    PADROES = {"esgoto_menos_lixo": contrafactual(MODELO, X, cat_sewer=1.0),
               # rajada acompanha o vento (no gerador, rajada = 1,8 × vento), então variam juntos
               "vento_30_menos_5": contrafactual(MODELO, X, clima_vento_kmh=30.0, clima_rajada_kmh=54.0) - contrafactual(MODELO, X, clima_vento_kmh=5.0, clima_rajada_kmh=9.0),
               "calor_32_menos_16": contrafactual(MODELO, X, clima_temperatura_c=32.0) - contrafactual(MODELO, X, clima_temperatura_c=16.0)}
    print({k: round(v, 2) for k, v in PADROES.items()})
    print("Verdade do gerador: esgoto−lixo = +1,4 | vento 30−5 = −0,88 | calor 32−16 = +0,8")
    print("O sinal e a ordem devem bater; o tamanho sai menor porque as notas são arredondadas e limitadas a 1–5.")
else:
    PADROES = None

### Prova das regras de retreino (só no modo sintético)

Três situações com resposta conhecida: poucos dados devem dar `nao_treinar`; notas embaralhadas (sem padrão
nenhum) devem dar `manter_vigente`; os dados completos devem dar `promover`.

In [ ]:
if VERDADE:
    pequenos = consenso.index[:12]
    d1, _ = decidir(df[df.grupo.isin(pequenos)], consenso.loc[pequenos], X.loc[pequenos], y[:12], peso[:12])
    embaralhado = np.random.default_rng(SEED).permutation(y)
    d2, _ = decidir(df, consenso, X, embaralhado, peso)
    PROVA_REGRAS = {"poucos_dados": d1["decisao"], "notas_embaralhadas": d2["decisao"], "dados_completos": DECISAO["decisao"]}
    print(PROVA_REGRAS)
    assert PROVA_REGRAS == {"poucos_dados": "nao_treinar", "notas_embaralhadas": "manter_vigente", "dados_completos": "promover"}
else:
    PROVA_REGRAS = None

## Salvar a versão

Toda rodada grava uma pasta com data e hora em `MODELOS_DIR`, mesmo quando não promove (fica o registro do
motivo). Só a promoção atualiza `vigente.json`. Os arquivos guardam pseudônimos, nunca nomes ou e-mails.

In [ ]:
import joblib, platform, sklearn

carimbo = dt.datetime.now().strftime("%Y-%m-%d_%H%M%S")
pasta = MODELOS_DIR / f"{carimbo}_{MODO}"
pasta.mkdir(parents=True, exist_ok=False)
registro = {"criado_em": carimbo, "modo": MODO, "dataset_sha256": DATASET_SHA, "relatos": int(len(df)),
            "participantes": int(df.participante.nunique()), "grupos": int(len(consenso)),
            "parametros": {"grupo_metros": GRUPO_METROS, "grupo_minutos": GRUPO_MINUTOS, "encolhimento_vies": ENCOLHIMENTO_VIES,
                           "margem": MARGEM_MELHORA, "fracao_dobras": FRACAO_DOBRAS_MELHORA,
                           "minimos": [MIN_RELATOS, MIN_PARTICIPANTES, MIN_GRUPOS, MIN_GRUPOS_MULTI]},
            "entradas": list(X.columns), "verificacao_sintetica": VERIF, "padroes_sinteticos": PADROES, "prova_regras": PROVA_REGRAS,
            "ambiente": {"python": platform.python_version(), "sklearn": sklearn.__version__, "seed": SEED}, **DECISAO}
(pasta / "registro.json").write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding="utf8")
consenso.to_csv(pasta / "consenso_grupos.csv")
vies.to_csv(pasta / "vies_participantes.csv")
df[["id", "grupo", "nota", "vies_estimado", "intensidade_ajustada"]].to_csv(pasta / "intensidade_ajustada.csv", index=False)
if MODELO is not None:
    joblib.dump(MODELO, pasta / "modelo.joblib")
    if MODO == "campo":   # modelo sintético nunca vira o vigente de campo
        vigente_path.write_text(json.dumps({"pasta": pasta.name, **registro}, ensure_ascii=False, indent=2), encoding="utf8")
print("Gravado em", pasta, "| decisão:", DECISAO["decisao"])

## Checklist semanal (durante a coleta)

1. **Exportar:** `MODO = "campo"`, `EXPORTAR = True` e *Executar tudo*. Entre com a sua conta Google quando pedir.
2. **Conferir** o `manifesto.json`: número de relatos, excluídos de teste e fotos.
3. O mesmo *Executar tudo* já roda consenso, viés e a decisão sobre o modelo.
4. **Ler a decisão:** `nao_treinar` (faltam dados — normal no começo), `manter_vigente` ou `promover`.
5. **Olhar o consenso:** grupos com dispersão alta (> 1 nível) merecem revisão das observações e fotos.
6. **Olhar o viés:** um participante com viés muito alto não está "errado"; o viés só ajusta a escala.
7. **Registrar** na planilha de acompanhamento: data, relatos, participantes, decisão e MAE.
8. Nenhuma previsão vai para o mapa público até a decisão de pesquisa sobre isso.